In [1]:
"""
ARMA(1,1) Forecasting Model
=============================
Real-time recursive forecasts of log real TTF NG prices.
r_t = c + φ_1·r_{t-1} + ε_t + θ_1·ε_{t-1}
MA term drops out for h > 1; iterated AR(1) used from h=2 onward.
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START  = "2015-01-01"
INPUT_FILE  = "Input_TTF_NG_Real_Average_Prices.xlsx"
OUTPUT_FILE = "Output_ARMA11_forecasts.xlsx"
MODEL_NAME  = "ARMA(1,1)"


In [3]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, parse_dates=["date"])
df = df[["date","price_real"]].sort_values("date").reset_index(drop=True)
df["log_price"] = np.log(df["price_real"])

In [4]:
# ── Helper: actual real price for a given year-month ─────────────────────────
def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["price_real"].values[0] if len(m) == 1 else np.nan

# ── Helper: iterated ARMA(1,1) forecasts ─────────────────────────────────────
def arma11_forecasts(history, horizons):
    model = ARIMA(history, order=(1, 0, 1)).fit()
    # forecast returns log-space predictions for steps 1..max(horizons)
    h_max  = max(horizons)
    f_log  = model.forecast(steps=h_max)   # array of length h_max
    return {h: f_log[h-1] for h in horizons}

In [5]:
# ── Main forecasting loop ─────────────────────────────────────────────────────
records = []
origins = df[df["date"] >= EVAL_START]["date"].tolist()

for origin_date in origins:
    history = df[df["date"] <= origin_date]["log_price"].values

    # Need at least 3 observations to fit ARMA(1,1)
    if len(history) < 3:
        continue

    try:
        forecasts = arma11_forecasts(history, HORIZONS)
    except Exception:
        continue

    for h in HORIZONS:
        actual_ym      = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
        forecast_level = np.exp(forecasts[h])
        actual_val     = get_actual(actual_ym)

        records.append({
            "forecast_origin": origin_date.strftime("%Y-%m-%d"),
            "horizon":         h,
            "model":           MODEL_NAME,
            "actual_month":    actual_ym,
            "forecast":        forecast_level,
            "actual":          actual_val,
        })

In [6]:
# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

In [7]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("ARMA(1,1) forecasting complete.")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Horizons:         {HORIZONS}")
print(f"  Total rows:       {len(results)}")
print(f"  Output saved to:  {OUTPUT_FILE}")
print()

ARMA(1,1) forecasting complete.
  Forecast origins: 132
  Horizons:         [1, 3, 6, 9, 12, 15, 18, 21, 24]
  Total rows:       1188
  Output saved to:  Output_ARMA11_forecasts.xlsx



In [8]:
# Sample — first origin
first = results[results["forecast_origin"] == results["forecast_origin"].min()]
print(f"Sample — first origin ({first['forecast_origin'].iloc[0]}):")
print(first[["horizon","actual_month","forecast","actual"]].to_string(index=False))

Sample — first origin (2015-01-31):
 horizon actual_month  forecast    actual
       1      2015-02 19.797793 22.938516
       3      2015-04 20.027621 22.046423
       6      2015-07 20.299907 20.679393
       9      2015-10 20.503868 18.160884
      12      2016-01 20.656185 13.882111
      15      2016-04 20.769677 12.101522
      18      2016-07 20.854099 14.115551
      21      2016-10 20.916819 15.958961
      24      2017-01 20.963374 19.873384
